# Ligand run analysisLoads one run directory and answers three questions:1. Is the output actually diverse, or does it look diverse because the property distributions are wide?2. Which candidates are worth a chemist's attention?3. Does docking score correlate with anything, or is it noise?Set `RUN_DIR` and run all.

In [ ]:
from pathlib import Pathimport jsonimport pandas as pdimport matplotlib.pyplot as pltRUN_DIR = Path("../runs/ligand_demo")run = json.loads((RUN_DIR / "run.json").read_text())df = pd.read_csv(RUN_DIR / "results.csv")print(f"target       {run['pocket'].get('metadata', {}).get('id')}")print(f"backend      {run['config']['generation']['backend']}")print(f"candidates   {len(df)}  |  valid {int(df['valid'].sum())}")print(f"verdict      {run['diversity_report'].get('verdict')}")df.head()

## 1. DiversityRead this before the property plots. Wide property distributions on a single scaffold are the v1 failure, and they look healthy on a histogram.

In [ ]:
rep = run["diversity_report"]for key in ["n_unique", "duplicate_rate", "n_scaffolds", "n_generic_scaffolds",            "scaffold_diversity", "scaffold_entropy", "largest_scaffold_share",            "internal_diversity", "mean_pairwise_tanimoto"]:    if key in rep:        print(f"{key:26s} {rep[key]}")print()print(rep.get("verdict"))

In [ ]:
hist = rep.get("tanimoto_histogram", {})if hist:    fig, ax = plt.subplots(figsize=(7, 3.5))    ax.bar(list(hist), list(hist.values()), color="#4a7fb5")    ax.set_xlabel("pairwise Tanimoto")    ax.set_ylabel("pairs")    ax.set_title("Similarity distribution — mass on the right means one chemotype")    plt.tight_layout()    plt.show()

In [ ]:
top = rep.get("top_scaffolds", [])if top:    labels = [s if len(s) < 30 else s[:27] + "..." for s, _ in top]    counts = [c for _, c in top]    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.35 * len(top))))    ax.barh(labels[::-1], counts[::-1], color="#5a9e78")    ax.set_xlabel("molecules")    ax.set_title("Scaffold occupancy")    plt.tight_layout()    plt.show()

## 2. PropertiesDashed lines are Lipinski / common practice thresholds.

In [ ]:
valid = df[df["valid"] == True]panels = [("mw", 500, "MW (Da)"), ("logp", 5, "cLogP"), ("tpsa", 140, "TPSA"),          ("hbd", 5, "HBD"), ("hba", 10, "HBA"), ("qed", None, "QED")]fig, axes = plt.subplots(2, 3, figsize=(14, 7))for ax, (col, cutoff, label) in zip(axes.ravel(), panels):    if col not in valid:        continue    ax.hist(valid[col].dropna(), bins=20, color="#4a7fb5", edgecolor="white")    if cutoff is not None:        ax.axvline(cutoff, color="#c0392b", linestyle="--", linewidth=1)    ax.set_xlabel(label)    ax.set_ylabel("count")plt.tight_layout()plt.show()

## 3. DockingScores rank poses. They are not affinities — treat them as triage.

In [ ]:
scored = valid.dropna(subset=["best_affinity"])if scored.empty:    print("no docking results in this run")else:    print(scored["best_affinity"].describe().round(2).to_string())    fig, axes = plt.subplots(1, 3, figsize=(15, 4))    axes[0].hist(scored["best_affinity"], bins=20, color="#5a9e78", edgecolor="white")    axes[0].axvline(-7, color="#c0392b", linestyle="--", linewidth=1)    axes[0].set_xlabel("Vina score (kcal/mol)")    axes[0].set_title("Score distribution")    axes[1].scatter(scored["mw"], scored["best_affinity"], alpha=.65, color="#4a7fb5")    axes[1].set_xlabel("MW (Da)")    axes[1].set_ylabel("Vina score")    axes[1].set_title("Score vs size — a strong trend means size bias, not affinity")    if "reference_rmsd" in scored and scored["reference_rmsd"].notna().any():        axes[2].scatter(scored["reference_rmsd"], scored["best_affinity"], alpha=.65, color="#8e6bb5")        axes[2].axvline(2.0, color="#c0392b", linestyle="--", linewidth=1)        axes[2].set_xlabel("RMSD to reference (A)")        axes[2].set_ylabel("Vina score")        axes[2].set_title("Score vs pose deviation")    else:        axes[2].axis("off")    plt.tight_layout()    plt.show()    corr = scored[["best_affinity", "mw", "logp", "tpsa", "qed"]].corr()["best_affinity"].round(2)    print("\ncorrelation with score:")    print(corr.drop("best_affinity").to_string())

## 4. Top candidates

In [ ]:
from rdkit import Chemfrom rdkit.Chem import Drawbest = scored.nsmallest(12, "best_affinity") if not scored.empty else valid.nlargest(12, "qed")mols, legends = [], []for _, row in best.iterrows():    mol = Chem.MolFromSmiles(row["canonical_smiles"])    if mol is None:        continue    mols.append(mol)    score = row.get("best_affinity")    legends.append(f"{row['name']}\n{score:.2f} kcal/mol" if pd.notna(score) else str(row["name"]))Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(260, 220), legends=legends)

## 5. ShortlistRanked on score, then filtered on everything a chemist would check anyway.

In [ ]:
shortlist = (    scored[(scored["passes_all"] == True) & (scored["best_affinity"] < -7.0) & (scored["qed"] > 0.4)]    .nsmallest(20, "best_affinity")    [["name", "canonical_smiles", "best_affinity", "mw", "logp", "tpsa", "qed", "sa_score"]])shortlist.to_csv(RUN_DIR / "shortlist.csv", index=False)print(f"{len(shortlist)} candidates -> {RUN_DIR / 'shortlist.csv'}")shortlist